<a href="https://colab.research.google.com/github/Pavun-KumarCH/AI-Enhanced-RAG-System-for-Automated-University-Course-Content-Generation/blob/main/RAG_VDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#@title requirements
%pip install --q openai pinecone datasets

In [6]:
# Load Dependecies
import os
import ast
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm
from datasets import load_dataset
from pinecone import Pinecone, ServerlessSpec
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings('ignore')

from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
PINECONE_API_KEY= userdata.get('PINECONE_API_KEY')

In [7]:
def create_dlai_index_name(index_name):
  openai_key = ""
  try:
    # For Google Colab
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
  except ImportError:
    # For Jupyter or other environments
    openai_key = os.getenv("OPENAI_API_KEY")

    # Ensure openai_key is not empty
  if not openai_key:
    raise ValueError("OpenAI API key is missing.")

  return f'{index_name}-{openai_key[-36:].lower().replace("_", "-")}'

In [25]:
# Setup Pinecone

pinecone = Pinecone(api_key = PINECONE_API_KEY)

INDEX_NAME = create_dlai_index_name("rag-dlai")
if INDEX_NAME in[index.name for index in pinecone.list_indexes()]:
  pinecone.delete_index(INDEX_NAME)

pinecone.create_index(
  name = INDEX_NAME,
  dimension = 1536,
  metric = "cosine",
  spec = ServerlessSpec(cloud = "aws", region = "us-east-1"))

# Creating a Index
index = pinecone.Index(INDEX_NAME)
index

In [12]:
# Download the Data

#!wget -q -O lesson2-wiki.csv.zip "https://www.dropbox.com/scl/fi/yxzmsrv2sgl249zcspeqb/lesson2-wiki.csv.zip?rlkey=paehnoxjl3s5x53d1bedt4pmc&dl=0"

#!unzip lesson2-wiki.csv.zip

In [27]:
#@title load The Dataset
max_articles_num = 500
df = pd.read_csv('wiki.csv', nrows = max_articles_num)
df.head()

,id,metadata,values
1,1-0,"{'chunk': 0, 'source': 'https://simple.wikiped...","[-0.011254455894231796, -0.01698738895356655, ..."
2,1-1,"{'chunk': 1, 'source': 'https://simple.wikiped...","[-0.0015197008615359664, -0.007858820259571075..."
3,1-2,"{'chunk': 2, 'source': 'https://simple.wikiped...","[-0.009930099360644817, -0.012211072258651257,..."
4,1-3,"{'chunk': 3, 'source': 'https://simple.wikiped...","[-0.011600767262279987, -0.012608098797500134,..."
5,1-4,"{'chunk': 4, 'source': 'https://simple.wikiped...","[-0.026462381705641747, -0.016362832859158516,..."


In [15]:
#@title Prepare the Embeddings and Upsert(upload) to Pinecone
prepared = []

for i, row in tqdm(df.iterrows(), total = df.shape[0]):
  meta = ast.literal_eval(row['metadata'])
  values = ast.literal_eval(row['values'])
  prepared.append({'id' : row['id'],
                   'values' : values,
                   'metadata': meta})
  if len(prepared) >= 250:
    index.upsert(vectors = prepared)
    prepared = []

  0%|          | 0/500 [00:00<?, ?it/s]

In [16]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 500}},
 'total_vector_count': 500}

In [17]:
#@title Connect to OpenAI
openai_client = OpenAI(api_key = OPENAI_API_KEY)

def get_embedding(articles, model = "text-embedding-ada-002"):
  return [record.embedding for record in openai_client.embeddings.create(input = articles, model = model).data]

In [20]:
#@title Run Your Query
query = "What is the berlin wall ?"

embed = get_embedding([query])
res = index.query(vector = embed, top_k = 4, include_metadata = True)
text = [r['metadata']['text'] for r in res['matches']]
print("\n".join(text))

August 13  1961: Building of the Berlin Wall begins.
 August 14  1945: Japan announces its surrender at the end of World War II.
 August 14/15  1947: India is partitioned at independence from the UK, as the new mainly Islamic state of Pakistan is created.
 August 15  1960: The Republic of the Congo becomes independent.
 August 15  1971: Bahrain becomes independent.
 August 16  1977: Elvis Presley dies aged 42, leading to a worldwide outpouring of grief.
 August 17  1945: Indonesia declares independence from the Netherlands.
 August 17  1960: Gabon becomes independent.
 August 17  1962: Peter Fechter becomes the first person to be shot dead at the Berlin Wall.
 August 19  43 BC: Augustus becomes Roman consul.
 August 19  14: Augustus dies.
 August 19  1919: Afghanistan becomes independent.
 August 19  1991: The August Coup against Mikhail Gorbachev, in the Soviet Union, begins.
 August 20  1940: Leon Trotsky is fatally wounded with an ice pick in Mexico.
 August 20  1968: The Prague Spr

In [22]:
## Build the Prompt
query = "write an article titled: what is the berlin wall?"
embed = get_embedding([query])

res = index.query(vector = embed,
                  top_k = 3,
                  include_metadata = True)

contexts = [x['metadata']['text'] for x in res['matches']]

prompt_start = ("Answer the question based on the context below.  \n\n"+
                "Context:\n")

prompt_end = (f"\n\nQuestion: {query}\nAnswer:")

prompt = (prompt_start + "\n\n---\n\n".join(contexts) + "\n\n---\n\n" + prompt_end)

print(prompt)

Answer the question based on the context below.  

Context:
August 13  1961: Building of the Berlin Wall begins.
 August 14  1945: Japan announces its surrender at the end of World War II.
 August 14/15  1947: India is partitioned at independence from the UK, as the new mainly Islamic state of Pakistan is created.
 August 15  1960: The Republic of the Congo becomes independent.
 August 15  1971: Bahrain becomes independent.
 August 16  1977: Elvis Presley dies aged 42, leading to a worldwide outpouring of grief.
 August 17  1945: Indonesia declares independence from the Netherlands.
 August 17  1960: Gabon becomes independent.
 August 17  1962: Peter Fechter becomes the first person to be shot dead at the Berlin Wall.
 August 19  43 BC: Augustus becomes Roman consul.
 August 19  14: Augustus dies.
 August 19  1919: Afghanistan becomes independent.
 August 19  1991: The August Coup against Mikhail Gorbachev, in the Soviet Union, begins.
 August 20  1940: Leon Trotsky is fatally wounded 

In [24]:
# Get the Summary
res = openai_client.completions.create(
    model = "gpt-3.5-turbo-instruct",
    prompt = prompt,
    temperature = 0.3,
    max_tokens = 636,
    top_p = 1,
    frequency_penalty = 0,
    presence_penalty = 0,
    stop = None)

print("-" * 80)
print(res.choices[0].text)
print("-" * 80)

--------------------------------------------------------------------------------


The Berlin Wall was a physical barrier that divided the city of Berlin, Germany from 1961 to 1989. It was built by the Soviet Union in order to prevent East Germans from fleeing to the West. The wall was made up of concrete walls, barbed wire, and guard towers, and stretched for 96 miles around the city.

The construction of the Berlin Wall began on August 13, 1961 and was a result of increasing tensions between the Soviet Union and the Western powers after World War II. The city of Berlin had been divided into four zones of occupation, with the Soviet Union controlling the eastern part and the United States, Great Britain, and France controlling the western part. As tensions rose, many East Germans began to flee to the West in search of better economic opportunities and freedom.

In order to stop this mass exodus, the Soviet Union decided to build a physical barrier between East and West Berlin. The wal